# Guitar Effect Switcher

Line-in audio is routed through the selectable hardware chain:

`Noise Gate -> Overdrive -> Distortion -> RAT Distortion -> Amp Simulator -> Cab IR -> EQ -> Reverb`


In [ ]:
import audio_lab_pynq as aud
from IPython.display import display, clear_output

ol = aud.AudioLabOverlay()

def configure_audio_path(volume=0xE7):
    # Line-in AUX -> ADC
    ol.codec.R4_RECORD_MIXER_LEFT_CONTROL_0 = 0x01
    ol.codec.R5_RECORD_MIXER_LEFT_CONTROL_1 = 0x05
    ol.codec.R6_RECORD_MIXER_RIGHT_CONTROL_0 = 0x01
    ol.codec.R7_RECORD_MIXER_RIGHT_CONTROL_1 = 0x05
    ol.codec.R19_ADC_CONTROL = 0x03

    # FPGA serial playback -> DAC -> playback mixers
    ol.codec.R58_SERIAL_INPUT_ROUTE_CONTROL = 0x01
    ol.codec.R59_SERIAL_OUTPUT_ROUTE_CONTROL = 0x01
    ol.codec.R36_DAC_CONTROL_0 = 0x03
    ol.codec.R22_PLAYBACK_MIXER_LEFT_CONTROL_0 = 0x21
    ol.codec.R24_PLAYBACK_MIXER_RIGHT_CONTROL_0 = 0x41

    # PYNQ-Z2 output jack path
    ol.codec.R26_PLAYBACK_LR_MIXER_LEFT_LINE_OUTPUT_CONTROL = 0x03
    ol.codec.R27_PLAYBACK_LR_MIXER_RIGHT_LINE_OUTPUT_CONTROL = 0x09
    ol.codec.R31_PLAYBACK_LINE_OUTPUT_LEFT_VOLUME_CONTROL = volume
    ol.codec.R32_PLAYBACK_LINE_OUTPUT_RIGHT_VOLUME_CONTROL = volume

    # Keep headphone output path enabled as well.
    ol.codec.R29_PLAYBACK_HEADPHONE_LEFT_VOLUME_CONTROL = volume
    ol.codec.R30_PLAYBACK_HEADPHONE_RIGHT_VOLUME_CONTROL = volume
    ol.codec.R35_PLAYBACK_POWER_MANAGEMENT = 0x03

    # Codec clocks/DSP
    ol.codec.R61_DSP_ENABLE = 0x01
    ol.codec.R62_DSP_RUN = 0x01
    ol.codec.R65_CLOCK_ENABLE_0 = 0x7F
    ol.codec.R66_CLOCK_ENABLE_1 = 0x03

configure_audio_path()
print("AudioLab overlay loaded. Output path configured.")


In [ ]:
def apply_effects(
    noise_gate_on=False, noise_gate_threshold=8,
    overdrive_on=False, overdrive_tone=65, overdrive_level=100, overdrive_drive=30,
    distortion_on=False, distortion_tone=65, distortion_level=100, distortion=25,
    rat_on=False, rat_filter=35, rat_level=100, rat_drive=55, rat_mix=100,
    amp_on=False, amp_input_gain=35, amp_bass=50, amp_middle=50, amp_treble=50,
    amp_presence=45, amp_resonance=35, amp_master=80, amp_character=35,
    cab_on=False, cab_mix=100, cab_level=100, cab_model=1, cab_air=50,
    eq_on=False, eq_low=100, eq_mid=100, eq_high=100,
    reverb_on=False, reverb_decay=30, reverb_tone=65, reverb_mix=20,
):
    words = ol.set_guitar_effects(
        noise_gate_on=noise_gate_on,
        noise_gate_threshold=noise_gate_threshold,
        overdrive_on=overdrive_on,
        overdrive_tone=overdrive_tone,
        overdrive_level=overdrive_level,
        overdrive_drive=overdrive_drive,
        distortion_on=distortion_on,
        distortion_tone=distortion_tone,
        distortion_level=distortion_level,
        distortion=distortion,
        rat_on=rat_on,
        rat_filter=rat_filter,
        rat_level=rat_level,
        rat_drive=rat_drive,
        rat_mix=rat_mix,
        amp_on=amp_on,
        amp_input_gain=amp_input_gain,
        amp_bass=amp_bass,
        amp_middle=amp_middle,
        amp_treble=amp_treble,
        amp_presence=amp_presence,
        amp_resonance=amp_resonance,
        amp_master=amp_master,
        amp_character=amp_character,
        cab_on=cab_on,
        cab_mix=cab_mix,
        cab_level=cab_level,
        cab_model=cab_model,
        cab_air=cab_air,
        eq_on=eq_on,
        eq_low=eq_low,
        eq_mid=eq_mid,
        eq_high=eq_high,
        reverb_on=reverb_on,
        reverb_decay=reverb_decay,
        reverb_tone=reverb_tone,
        reverb_mix=reverb_mix,
    )
    active = [
        name for name, on in [
            ("Noise Gate", noise_gate_on),
            ("Overdrive", overdrive_on),
            ("Distortion", distortion_on),
            ("RAT Distortion", rat_on),
            ("Amp Simulator", amp_on),
            ("Cab IR", cab_on),
            ("EQ", eq_on),
            ("Reverb", reverb_on),
        ] if on
    ]
    route = " -> ".join(["Line-in"] + active + ["Output"]) if active else "Line-in -> Output"
    return words, route

try:
    import ipywidgets as widgets
except ImportError:
    print("ipywidgets is not available; call apply_effects(...) manually.")
else:
    style = {"description_width": "120px"}
    slider_layout = widgets.Layout(width="420px")

    noise_gate_on = widgets.Checkbox(value=False, description="Noise Gate", indent=False)
    overdrive_on = widgets.Checkbox(value=False, description="Overdrive", indent=False)
    distortion_on = widgets.Checkbox(value=False, description="Distortion", indent=False)
    rat_on = widgets.Checkbox(value=False, description="RAT Distortion", indent=False)
    amp_on = widgets.Checkbox(value=False, description="Amp Simulator", indent=False)
    cab_on = widgets.Checkbox(value=False, description="Cab IR", indent=False)
    eq_on = widgets.Checkbox(value=False, description="EQ", indent=False)
    reverb_on = widgets.Checkbox(value=False, description="Reverb", indent=False)

    gate_threshold = widgets.IntSlider(value=8, min=0, max=100, step=1, description="Gate threshold", style=style, layout=slider_layout)
    od_drive = widgets.IntSlider(value=30, min=0, max=100, step=1, description="OD drive", style=style, layout=slider_layout)
    od_tone = widgets.IntSlider(value=65, min=0, max=100, step=1, description="OD tone", style=style, layout=slider_layout)
    od_level = widgets.IntSlider(value=100, min=0, max=200, step=1, description="OD level", style=style, layout=slider_layout)
    dist_amount = widgets.IntSlider(value=25, min=0, max=100, step=1, description="Distortion", style=style, layout=slider_layout)
    dist_tone = widgets.IntSlider(value=65, min=0, max=100, step=1, description="Dist tone", style=style, layout=slider_layout)
    dist_level = widgets.IntSlider(value=100, min=0, max=200, step=1, description="Dist level", style=style, layout=slider_layout)
    rat_drive = widgets.IntSlider(value=55, min=0, max=100, step=1, description="RAT drive", style=style, layout=slider_layout)
    rat_filter = widgets.IntSlider(value=35, min=0, max=100, step=1, description="RAT filter", style=style, layout=slider_layout)
    rat_level = widgets.IntSlider(value=100, min=0, max=150, step=1, description="RAT level", style=style, layout=slider_layout)
    rat_mix = widgets.IntSlider(value=100, min=0, max=100, step=1, description="RAT mix", style=style, layout=slider_layout)
    amp_input_gain = widgets.IntSlider(value=35, min=0, max=100, step=1, description="Amp gain", style=style, layout=slider_layout)
    amp_bass = widgets.IntSlider(value=50, min=0, max=100, step=1, description="Amp bass", style=style, layout=slider_layout)
    amp_middle = widgets.IntSlider(value=50, min=0, max=100, step=1, description="Amp middle", style=style, layout=slider_layout)
    amp_treble = widgets.IntSlider(value=50, min=0, max=100, step=1, description="Amp treble", style=style, layout=slider_layout)
    amp_presence = widgets.IntSlider(value=45, min=0, max=100, step=1, description="Presence", style=style, layout=slider_layout)
    amp_resonance = widgets.IntSlider(value=35, min=0, max=100, step=1, description="Resonance", style=style, layout=slider_layout)
    amp_master = widgets.IntSlider(value=80, min=0, max=150, step=1, description="Master", style=style, layout=slider_layout)
    amp_character = widgets.IntSlider(value=35, min=0, max=100, step=1, description="Character", style=style, layout=slider_layout)
    cab_mix = widgets.IntSlider(value=100, min=0, max=100, step=1, description="Cab mix", style=style, layout=slider_layout)
    cab_level = widgets.IntSlider(value=100, min=0, max=150, step=1, description="Cab level", style=style, layout=slider_layout)
    cab_model = widgets.Dropdown(options=[("1x12 open", 0), ("2x12 british", 1), ("4x12 closed", 2)], value=1, description="Cab model", style=style, layout=slider_layout)
    cab_air = widgets.IntSlider(value=50, min=0, max=100, step=1, description="Cab air", style=style, layout=slider_layout)
    eq_low = widgets.IntSlider(value=100, min=0, max=200, step=1, description="EQ low", style=style, layout=slider_layout)
    eq_mid = widgets.IntSlider(value=100, min=0, max=200, step=1, description="EQ mid", style=style, layout=slider_layout)
    eq_high = widgets.IntSlider(value=100, min=0, max=200, step=1, description="EQ high", style=style, layout=slider_layout)
    reverb_decay = widgets.IntSlider(value=30, min=0, max=100, step=1, description="Reverb decay", style=style, layout=slider_layout)
    reverb_tone = widgets.IntSlider(value=65, min=0, max=100, step=1, description="Reverb tone", style=style, layout=slider_layout)
    reverb_mix = widgets.IntSlider(value=20, min=0, max=100, step=1, description="Reverb mix", style=style, layout=slider_layout)

    auto_apply = widgets.Checkbox(value=True, description="Apply on change", indent=False)
    apply_button = widgets.Button(description="Apply", button_style="primary")
    bypass_button = widgets.Button(description="Bypass")
    crunch_button = widgets.Button(description="Crunch")
    lead_button = widgets.Button(description="Lead")
    space_button = widgets.Button(description="Space")
    output = widgets.Output()

    controls = [
        noise_gate_on, overdrive_on, distortion_on, rat_on, amp_on, cab_on, eq_on, reverb_on,
        gate_threshold, od_drive, od_tone, od_level,
        dist_amount, dist_tone, dist_level,
        rat_drive, rat_filter, rat_level, rat_mix,
        amp_input_gain, amp_bass, amp_middle, amp_treble, amp_presence, amp_resonance, amp_master, amp_character,
        cab_mix, cab_level, cab_model, cab_air,
        eq_low, eq_mid, eq_high,
        reverb_decay, reverb_tone, reverb_mix,
    ]

    def params():
        return dict(
            noise_gate_on=noise_gate_on.value,
            noise_gate_threshold=gate_threshold.value,
            overdrive_on=overdrive_on.value,
            overdrive_tone=od_tone.value,
            overdrive_level=od_level.value,
            overdrive_drive=od_drive.value,
            distortion_on=distortion_on.value,
            distortion_tone=dist_tone.value,
            distortion_level=dist_level.value,
            distortion=dist_amount.value,
            rat_on=rat_on.value,
            rat_filter=rat_filter.value,
            rat_level=rat_level.value,
            rat_drive=rat_drive.value,
            rat_mix=rat_mix.value,
            amp_on=amp_on.value,
            amp_input_gain=amp_input_gain.value,
            amp_bass=amp_bass.value,
            amp_middle=amp_middle.value,
            amp_treble=amp_treble.value,
            amp_presence=amp_presence.value,
            amp_resonance=amp_resonance.value,
            amp_master=amp_master.value,
            amp_character=amp_character.value,
            cab_on=cab_on.value,
            cab_mix=cab_mix.value,
            cab_level=cab_level.value,
            cab_model=cab_model.value,
            cab_air=cab_air.value,
            eq_on=eq_on.value,
            eq_low=eq_low.value,
            eq_mid=eq_mid.value,
            eq_high=eq_high.value,
            reverb_on=reverb_on.value,
            reverb_decay=reverb_decay.value,
            reverb_tone=reverb_tone.value,
            reverb_mix=reverb_mix.value,
        )

    def update_output(*_):
        words, route = apply_effects(**params())
        with output:
            clear_output(wait=True)
            print(route)
            for key in ["gate", "overdrive", "distortion", "rat", "amp", "amp_tone", "cab", "eq", "reverb"]:
                print("%10s: 0x%08x" % (key, words[key]))

    def on_control_change(change):
        if auto_apply.value and change.get("name") == "value":
            update_output()

    def set_values(values):
        old_auto = auto_apply.value
        auto_apply.value = False
        for widget, value in values.items():
            widget.value = value
        auto_apply.value = old_auto
        update_output()

    def bypass(_):
        set_values({noise_gate_on: False, overdrive_on: False, distortion_on: False, rat_on: False, amp_on: False, cab_on: False, eq_on: False, reverb_on: False})

    def crunch(_):
        set_values({noise_gate_on: True, overdrive_on: True, distortion_on: False, rat_on: False, amp_on: True, cab_on: True, eq_on: True, reverb_on: False, od_drive: 45, od_tone: 70, od_level: 105, amp_input_gain: 42, amp_bass: 54, amp_middle: 58, amp_treble: 56, amp_presence: 46, amp_resonance: 38, amp_master: 85, amp_character: 40, cab_mix: 100, cab_level: 100, cab_model: 1, cab_air: 55, eq_low: 105, eq_mid: 115, eq_high: 115})

    def lead(_):
        set_values({noise_gate_on: True, overdrive_on: False, distortion_on: False, rat_on: True, amp_on: True, cab_on: True, eq_on: True, reverb_on: True, rat_drive: 65, rat_filter: 32, rat_level: 85, rat_mix: 85, amp_input_gain: 62, amp_bass: 48, amp_middle: 62, amp_treble: 58, amp_presence: 58, amp_resonance: 42, amp_master: 78, amp_character: 62, cab_mix: 100, cab_level: 95, cab_model: 2, cab_air: 60, eq_low: 95, eq_mid: 118, eq_high: 112, reverb_decay: 25, reverb_tone: 70, reverb_mix: 16})

    def space(_):
        set_values({noise_gate_on: True, overdrive_on: True, distortion_on: False, rat_on: True, amp_on: True, cab_on: True, eq_on: True, reverb_on: True, od_drive: 30, od_tone: 60, od_level: 90, rat_drive: 42, rat_filter: 55, rat_level: 78, rat_mix: 40, amp_input_gain: 38, amp_bass: 52, amp_middle: 50, amp_treble: 54, amp_presence: 42, amp_resonance: 45, amp_master: 75, amp_character: 35, cab_mix: 90, cab_level: 95, cab_model: 2, cab_air: 60, eq_low: 100, eq_mid: 95, eq_high: 108, reverb_decay: 45, reverb_tone: 75, reverb_mix: 28})

    for control in controls:
        control.observe(on_control_change, names="value")
    apply_button.on_click(update_output)
    bypass_button.on_click(bypass)
    crunch_button.on_click(crunch)
    lead_button.on_click(lead)
    space_button.on_click(space)

    toggles = widgets.HBox([noise_gate_on, overdrive_on, distortion_on, rat_on, amp_on, cab_on, eq_on, reverb_on])
    presets = widgets.HBox([apply_button, bypass_button, crunch_button, lead_button, space_button, auto_apply])
    accordion = widgets.Accordion(children=[
        widgets.VBox([gate_threshold]),
        widgets.VBox([od_drive, od_tone, od_level]),
        widgets.VBox([dist_amount, dist_tone, dist_level]),
        widgets.VBox([rat_drive, rat_filter, rat_level, rat_mix]),
        widgets.VBox([amp_input_gain, amp_bass, amp_middle, amp_treble, amp_presence, amp_resonance, amp_master, amp_character]),
        widgets.VBox([cab_mix, cab_level, cab_model, cab_air]),
        widgets.VBox([eq_low, eq_mid, eq_high]),
        widgets.VBox([reverb_decay, reverb_tone, reverb_mix]),
    ])
    for i, title in enumerate(["Noise Gate", "Overdrive", "Distortion", "RAT Distortion", "Amp Simulator", "Cab IR", "EQ", "Reverb"]):
        accordion.set_title(i, title)

    display(widgets.VBox([toggles, presets, accordion, output]))
    update_output()
